# Pro Cycling Intelligence Agent — Playground

Test every phase of the agent without Telegram. Each cell is independent so you can inspect exactly what happens at each step.

```
Question → [Planner] → ResearchPlan → [Executor] → Raw Findings → [Synthesizer] → CyclingAnswer
```

In [ ]:
# Run once to install all dependencies
%pip install openai firecrawl-py tavily-python httpx beautifulsoup4 pydantic python-dotenv rich

---
## Step 0 — Setup

Keys are loaded automatically from the `.env` file in this folder.
You don't need `TELEGRAM_BOT_TOKEN` for the notebook — only `GROQ_API_KEY`, `TAVILY_API_KEY`, and `FIRECRAWL_API_KEY`.

In [2]:
import os
import sys
from dotenv import load_dotenv, find_dotenv

# Add the project root to path so imports work regardless of where Jupyter was launched
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# find_dotenv() walks up the directory tree to find the nearest .env
load_dotenv(find_dotenv())

groq_ok      = bool(os.environ.get("GROQ_API_KEY"))
tavily_ok    = bool(os.environ.get("TAVILY_API_KEY"))
firecrawl_ok = bool(os.environ.get("FIRECRAWL_API_KEY"))

print(f"GROQ_API_KEY      : {'OK' if groq_ok      else 'MISSING — check your .env'}")
print(f"TAVILY_API_KEY    : {'OK' if tavily_ok    else 'MISSING — check your .env'}")
print(f"FIRECRAWL_API_KEY : {'OK' if firecrawl_ok else 'MISSING — check your .env'}")

GROQ_API_KEY      : OK
TAVILY_API_KEY    : OK
FIRECRAWL_API_KEY : OK


---
## Step 1 — The Data Models

Two schemas drive the whole agent:
- `ResearchPlan` — what the Planner outputs (which tools to call and with what queries)
- `CyclingAnswer` — what the Synthesizer outputs (the final structured answer)

Both are passed to **Llama 3.3 70b** (via Groq) as tool definitions. The model fills them in like a form — no free-text parsing needed.

In [2]:
from models.plan import ResearchPlan, CyclingToolType
from models.answer import CyclingAnswer
import json

print("=== CyclingToolType — all available tools ===")
for t in CyclingToolType:
    print(f"  {t.value}")

print()
print("=== ResearchPlan schema (sent to Claude as a tool) ===")
print(json.dumps(ResearchPlan.model_json_schema(), indent=2))

=== CyclingToolType — all available tools ===
  pcs_ranking
  pcs_rider
  pcs_race
  pcs_stage
  pcs_startlist
  pcs_rider_results
  search
  scrape
  firecrawl

=== ResearchPlan schema (sent to Claude as a tool) ===
{
  "$defs": {
    "CyclingToolType": {
      "enum": [
        "pcs_ranking",
        "pcs_rider",
        "pcs_race",
        "pcs_stage",
        "pcs_startlist",
        "pcs_rider_results",
        "search",
        "scrape",
        "firecrawl"
      ],
      "title": "CyclingToolType",
      "type": "string"
    },
    "ResearchStep": {
      "properties": {
        "step_id": {
          "title": "Step Id",
          "type": "integer"
        },
        "description": {
          "description": "What this step is trying to find out",
          "title": "Description",
          "type": "string"
        },
        "tool": {
          "$ref": "#/$defs/CyclingToolType",
          "description": "Which tool to use"
        },
        "query": {
          "description": 

In [3]:
print("=== CyclingAnswer schema (sent to Claude as a tool) ===")
print(json.dumps(CyclingAnswer.model_json_schema(), indent=2))

=== CyclingAnswer schema (sent to Claude as a tool) ===
{
  "properties": {
    "question": {
      "description": "The original user question",
      "title": "Question",
      "type": "string"
    },
    "answer": {
      "description": "Main answer in clear prose, 3-6 sentences",
      "title": "Answer",
      "type": "string"
    },
    "data_points": {
      "description": "Key facts as short bullet points",
      "items": {
        "type": "string"
      },
      "title": "Data Points",
      "type": "array"
    },
    "source_note": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Where the data came from",
      "title": "Source Note"
    },
    "follow_up_suggestions": {
      "description": "2-3 related questions the user might ask next",
      "items": {
        "type": "string"
      },
      "title": "Follow Up Suggestions",
      "type": "array"
    },
    

---
## Step 2 — The PCS Tools (no AI)

These are plain Python functions that search the web via Tavily for structured cycling data.
No Claude, no randomness. Test them directly here to see the raw data before the agent processes it.

In [3]:
from tools.cycling_pcs import get_individual_ranking, get_team_ranking

print("=== Top 5 UCI WorldTour individual ranking ===")
ranking = get_individual_ranking(top_n=5)
for r in ranking:
    print(r)

=== Top 5 UCI WorldTour individual ranking ===
{'title': 'UCI Ranking 2026-2028', 'url': 'https://www.procyclingstats.com/rankings/me/ranking-2026-2028', 'content': '+ Team Visma | Lease a Bike. # UCI World Ranking. ## Ranking 2026-2028. Team ranking for the 2026-2028 period on which WorldTour licenses are being assigned. Points are computed by the consolidated points of the 20 best riders per team in each year. There are 18 WorldTour licenses to be distributed. | 5 | - | XDS Astana Team | WT | 7722 | 7722 |. | 15 | - | Pinarello Q36.5 Pro Cycling Team | PRT | 4250 | 4250 |. | 17 | - | NSN Cycling Team | WT | 4162 | 4162 |. | 21 | - | Tudor Pro Cycling Team | PRT | 3538 | 3538 |. | 32 | - | Team Flanders - Baloise | PRT | 588 | 588 |. | 33 | - | Modern Adventure Pro Cycling | PRT | 565 |  |. | 34 | - | Team Novo Nordisk | PRT | 3 |  |.'}
{'title': 'Ranking 2026 - CyclingRanking.com', 'url': 'https://www.cyclingranking.com/riders/currentyear', 'content': '#### Search Team. | Tadej POGAČ

In [4]:
from tools.cycling_pcs import get_rider_profile

# Change the slug to any rider — format: firstname-lastname
rider_slug = "tadej-pogacar"
print(f"=== Rider profile: {rider_slug} ===")
profile = get_rider_profile(rider_slug)
print(json.dumps(profile, indent=2, default=str))

=== Rider profile: tadej-pogacar ===
{
  "results": [
    {
      "title": "Tadej Pogacar | Professional cyclist | Profile | News | Successes | TOUR",
      "url": "https://www.tour-magazin.de/en/driver-old/tadej-pogacar/",
      "content": "# Tadej Pogacar. Tadej Pogacar is a Slovenian professional cyclist who is under contract with UAE Team Emirates from the United Arab Emirates. The most important facts and figures about the two-time Tour de France champion. * Overall victory Tour de France 2020 & 2021. * 10 stage wins at the Tour de France (2020, 2021, 2022 & 2023). * Mountain classification of the Tour de France 2020 & 2021. * White jersey for the best young professional at the Tour de France 2020, 2021 & 2022. * 1st place Tour of Lombardy 2021, 2022 & 2023. * 1st place in the Tour of Flanders 2023. * Overall victory UAE Tour 2021 & 2022. The greatest successes in the Slovenian's still young career are the two overall victories in the Tour de France 2020 and 2021. Following his To

In [5]:
from tools.cycling_pcs import get_race_overview

# Format: get_race_overview(race_slug, year)
race = get_race_overview("tour-de-france", 2026)
print("=== Tour de France 2024 overview ===")
print(json.dumps(race, indent=2, default=str))

=== Tour de France 2024 overview ===
{
  "results": [
    {
      "title": "Tour de France 2026: Results",
      "url": "https://www.cyclingstage.com/tour-de-france-2026-results/",
      "content": "# Tour de France 2026: Results. Giro**The 2026 Tour de France sets off from Barcelona, Spain, on 4 July and the race will finish in Paris on 26 July. The 113th edition of the La Grande Boucle features more than three weeks of racing and 21 stages. Day in day out you\u2019ll find race results, stage reports and videos at Cyclingstage.com.**. As soon as stage 1 finishes in Barcelona, you'll read about the race results at this page. Tadej Pogacar is title holder. The Slovenian won last year's Tour de France ahead of Jonas Vingegaard and Florian Lipowitz. Another interesting read: records & winners Tour de France."
    },
    {
      "title": "Tour de France 2026 | Stage winners and leaders",
      "url": "https://www.procyclingstats.com/race/tour-de-france/2026/results",
      "content": "+ La

In [6]:
from tools.cycling_pcs import get_stage_results

# Format: get_stage_results(race_slug, year, stage_number)
stage = get_stage_results("tour-de-france", 2025, 5)
print("=== Tour de France 2025 — Stage 5 results ===")
print(json.dumps(stage, indent=2, default=str))

=== Tour de France 2025 — Stage 5 results ===
{
  "stage": 5,
  "results": [
    {
      "title": "Tour de France 2025 Stage 5 (ITT) results",
      "url": "https://www.procyclingstats.com/race/tour-de-france/2025/stage-5/result/result",
      "content": "+ Team Visma | Lease a Bike. Rnk | GC | Timelag | BIB | H2H | Specialty | Age | Rider | Team | UCI | Pnt | Time | Avg || 1 | 2 | +0:42 | 21 | TT | 25 | Evenepoel Remco Soudal Quick-Step | Soudal Quick-Step | 210 | 100 | 36.42,02 | 53.950 |. | 2 | 1 | +0:00 | 1 | Climber | 26 | Poga\u010dar Tadej UAE Team Emirates - XRG fav\\_gc | UAE Team Emirates - XRG | 150 | 70 | 0.16,68 | 53.545 |. | 11 | 5 | +1:22 | 15 | GC | 26 | Jorgenson Matteo Team Visma | Lease a Bike fav\\_gc | Team Visma | Lease a Bike | 25 | 8 | 1.19,2 | 52.077 |. | 16 | 107 | +23:46 | 4 | Classic | 31 | Politt Nils UAE Team Emirates - XRG | UAE Team Emirates - XRG | 1.37,64 | 51.660 |."
    },
    {
      "title": "2025 Tour de France Stage 5 Results: Remco Evenepoel\u20

In [8]:
from tools.cycling_pcs import get_rider_results

# Format: get_rider_results(rider_slug, year)
results = get_rider_results("tadej-pogacar", 2026)
print("=== Pogacar 2026 results (first 5) ===")
for r in results[:5]:
    print(r)

=== Pogacar 2026 results (first 5) ===
{'title': 'Tadej Pogacar Results & Career: A Complete Breakdown (2026)\n      \n      \n       – Bikeroom', 'url': 'https://bike-room.com/blogs/news/tadej-pogacar-results-analyzing-his-career-and-2026-season', 'content': "# Tadej Pogacar Results: Analyzing His Career and 2026 Season. This article reviews his 2026 spring campaign, his dominant 2025 season, and the broader context of the **Pogacar career**. The 2026 season opened with a specific focus on the major spring one-day races. On March 7, 2026, Pogačar secured his fourth victory at Strade Bianche. March 22, 2026, marked a significant addition to the **Pogacar career** palmarès: his first win at Milano-Sanremo. Continuing his spring block, Pogačar took his third career victory at the Tour of Flanders on April 5. **What are the most notable Tadej Pogacar results?** His major career results include four Tour de France victories (2020, 2021, 2024, 2025), a Giro d'Italia overall win (2024), Worl

### Web search fallback
Used when PCS doesn't have the data — live race updates, breaking news, etc.

In [7]:
from tools.web_search import search

results = search("Tour de France 2025 stage results latest", max_results=3)
for r in results:
    print(f"{r.title}\n{r.url}\n{r.content[:200]}\n")

Tour de France 2025 Stage 21 results - Pro Cycling Stats
https://www.procyclingstats.com/race/tour-de-france/2025/gc
Tadej Pogačar is the winner of Tour de France 2025, before Jonas Vingegaard and Florian Lipowitz. The stage from Mantes-la-Ville to Paris (Champs-Élysées)

2025 Tour de France - Wikipedia
https://en.wikipedia.org/wiki/2025_Tour_de_France
Winner, Tadej Pogačar ( SLO ), (UAE Team Emirates XRG). Second, Jonas Vingegaard ( DEN ), (Visma–Lease a Bike). Third, Florian Lipowitz ( GER ), (Red Bull–Bora–

Tour de France 2025 - Overall Individual Classification - Cyclingflash
https://cyclingflash.com/race/tour-de-france-2025/result/stage-21
1. BE flag Wout van Aert · Team Visma | Lease a Bike · #16 ; 2. IT flag Davide Ballerini · XDS Astana Team · #406 ; 3. SI flag Matej Mohorič · Bahrain Victorious.



### Firecrawl — live JS-rendered pages
Tavily searches the web. Firecrawl *renders* a specific page with a headless browser and returns clean markdown.
Use it when you need the actual table from procyclingstats.com instead of a search snippet.

**Requires `FIRECRAWL_API_KEY` in your `.env`** — get a free key at https://firecrawl.dev (500 pages/month).

In [ ]:
from tools.live_scraper import scrape_pcs_ranking, scrape_pcs_stage, scrape_pcs_gc, scrape_pcs_live

# WorldTour ranking — rendered directly from procyclingstats.com
print("=== WorldTour individual ranking (live from PCS) ===")
ranking = scrape_pcs_ranking()
print(ranking[:1500])

In [4]:
# Specific stage results — rendered from the actual PCS stage page
print("=== Giro 2026 Stage 10 (live from PCS) ===")
stage = scrape_pcs_stage("giro-d-italia", 2026, 10)
print(stage[:1500])

=== Giro 2026 Stage 10 (live from PCS) ===
| Rnk | GC | Timelag | BIB | H2H | Specialty | Age | Rider | Team |  | Avg |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |

Timelimit 33%

#### Race information

- Date:

19 May 2026

- Start time:

13:15

- Avg. speed winner:

  -

- Classification:

2.UWT

- Race category:

ME - Men Elite

- Distance:

42 km

- Points scale:

[GT.B.Stage](https://www.procyclingstats.com/info.php?s=point-scales&season=2026&category=1&scale=8)

- UCI scale:

[UCI.WR.GT.B.Stage](https://www.procyclingstats.com/info.php?s=point-scales&season=2026&category=1&scale=9863)

- Parcours type:

- Gradient final km:

0.1%

- ProfileScore:

2

- Vertical meters:

105

- Departure:

[Viareggio](https://www.procyclingstats.com/location/viareggio)

- Arrival:

[Massa](https://www.procyclingstats.com/location/massa)

- Race ranking:

[4](https://www.procyclingstats.com/rankings/me/races)

- Startlist quality score:

[955](https://www.procyclingstats.com

In [9]:
# GC standings for an ongoing race — great for "who is leading?"
print("=== Giro 2026 GC standings (live from PCS) ===")
gc = scrape_pcs_gc("giro-d-italia", 2026)
print(gc[:1500])

=== Giro 2026 GC standings (live from PCS) ===
| Rnk | GC | Timelag | BIB | H2H | Specialty | Age | Rider | Team |  |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |

| Rnk | BIB | H2H | Specialty | Age | Rider | Team |  | Time won/lost |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |

- [General](https://www.procyclingstats.com/race.php)
- [Today](https://www.procyclingstats.com/race.php)

| Rnk | BIB | H2H | Specialty | Age | Rider | Team | Today |
| --- | --- | --- | --- | --- | --- | --- | --- |

#### Sprint \| Fontana dello Zodiaco (23.1 km)

| Rnk | BIB | H2H | Specialty | Age | Rider | Team | Pnt | Today |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 |  |  |  |  |  |  | 12 |  |
| 2 |  |  |  |  |  |  | 8 |  |
| 3 |  |  |  |  |  |  | 5 |  |
| 4 |  |  |  |  |  |  | 3 |  |
| 5 |  |  |  |  |  |  | 1 |  |

#### Red Bull KM (77.7 km)

| Rnk | BIB | H2H | Specialty | Age | Rider | Team | Pnt | Bonis | Today |
| --- | --- | --- | --- | --- | --- | --

In [ ]:
from tools.live_scraper import scrape_pcs_stage_live

# Live ticker for a specific stage — URL: /race/{race}/year/stage-{N}/live
# Change the stage number to whichever stage is currently running
print("=== Giro 2026 Stage 10 — LIVE ticker ===")
live = scrape_pcs_stage_live("giro-d-italia", 2026, 11)
print(live[:10000])

---
## Step 3 — The Planner

Claude reads your question and decides which tools to call and in what order.
It outputs a structured `ResearchPlan` by being forced to call a tool — no free text.

**Change the question here:**

In [9]:
QUESTION = "Who is currently leading the WorldTour standings?"  # <- change this

In [10]:
from agent.planner import create_plan

print(f"Question: {QUESTION}\n")
plan = create_plan(QUESTION)

print(f"Claude generated {len(plan.steps)} steps:\n")
for step in plan.steps:
    print(f"  Step {step.step_id} [{step.tool.value}]")
    print(f"    Goal : {step.description}")
    print(f"    Query: {step.query}")
    print()

Question: Who is currently leading the WorldTour standings?

Claude generated 1 steps:

  Step 1 [pcs_ranking]
    Goal : Fetch the current UCI WorldTour individual standings to find the leader
    Query: 20



**Notice:** For a cycling question, the model picks PCS tools over web search when possible — structured data is more reliable than scraped text. Run the cell twice and the steps may vary slightly (LLMs are non-deterministic).

---
## Step 4 — The Executor

A plain `for` loop — no AI. Reads each step, calls the right tool, collects raw data.
The output is a dict: `{ topic description → raw JSON or text }`

In [11]:
from agent.executor import execute_plan

print(f"Executing {len(plan.steps)} steps...\n")
raw_findings = execute_plan(plan)

print("Topics collected:")
for topic in raw_findings:
    print(f"  - {topic}")

Executing 1 steps...

Topics collected:
  - Fetch the current UCI WorldTour individual standings to find the leader


In [12]:
# Inspect raw data for any topic — change [0] to explore others
topic = list(raw_findings.keys())[0]
print(f"=== Raw findings for: '{topic}' ===\n")
print(raw_findings[topic][:2000])

=== Raw findings for: 'Fetch the current UCI WorldTour individual standings to find the leader' ===

| # | Prev. | Diff. | H2H | [h2h](https://www.procyclingstats.com/rankings.php "Toggle a quick head-to-head column") Rider | Team | Points |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1 | - | [H2H](https://www.procyclingstats.com/rankings.php) [goto](https://www.procyclingstats.com/rankings.php) | [POGAČAR Tadej](https://www.procyclingstats.com/rider/tadej-pogacar) | [UAE Team Emirates - XRG](https://www.procyclingstats.com/team/uae-team-emirates-xrg-2026) | [4772](https://www.procyclingstats.com/rider.php?date=2026-05-18&id=194619&p=results&s=pcs-ranking) |
| 2 | 2 | - | [H2H](https://www.procyclingstats.com/rankings.php) [goto](https://www.procyclingstats.com/rankings.php) | [DEL TORO Isaac](https://www.procyclingstats.com/rider/isaac-del-toro) | [UAE Team Emirates - XRG](https://www.procyclingstats.com/team/uae-team-emirates-xrg-2026) | [3068](https://www.procyclingstats.com/r

---
## Step 5 — The Synthesizer

Claude reads all raw findings and produces a typed `CyclingAnswer`.
Same tool-use trick as the Planner — Claude fills in the schema fields, Pydantic validates them.

In [12]:
from agent.synthesizer import synthesize

print("Synthesizing answer...\n")
answer = synthesize(QUESTION, raw_findings)

print("Raw JSON output from Claude:")
print(answer.model_dump_json(indent=2))

Synthesizing answer...

Raw JSON output from Claude:
{
  "question": "Who is currently leading the WorldTour standings?",
  "answer": "Tadej Pogačar (UAE Team Emirates - XRG) is currently leading the UCI Individual WorldTour standings by a commanding margin. The Slovenian superstar has accumulated 11,630 points, putting him well clear of second-placed Jonas Vingegaard (Team Visma | Lease a Bike) on 6,885 points. Isaac Del Toro, also of UAE Team Emirates - XRG, sits third with 6,711 points, underlining the strength of the UAE outfit at the top of the rankings. Pogačar's dominance at the summit is no surprise given his recent exceptional form across multiple race categories.",
  "data_points": [
    "1st: Tadej Pogačar (UAE Team Emirates - XRG) — 11,630 points",
    "2nd: Jonas Vingegaard (Team Visma | Lease a Bike) — 6,885 points",
    "3rd: Isaac Del Toro (UAE Team Emirates - XRG) — 6,711 points",
    "7th: Paul Seixas (Decathlon CMA CGM Team) — 3,520 points",
    "8th: Wout van Aert (

---
## Step 6 — Rendered Report

The same output rendered as a readable report — this is what the Telegram bot sends (minus the HTML tags).

In [13]:
from IPython.display import display, Markdown

def report(a) -> None:
    confidence_icon = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(a.confidence, "⚪")
    data_points = "\n".join(f"- {p}" for p in a.data_points) or "_None_"
    follow_ups  = "\n".join(f"{i}. {s}" for i, s in enumerate(a.follow_up_suggestions, 1)) or "_None_"
    source      = f"_{a.source_note}_" if a.source_note else ""

    md = f"""
---
## 🚴 {a.question}

{a.answer}

### 📊 Key Facts
{data_points}

### 💡 You might also ask
{follow_ups}

**Confidence:** {confidence_icon} {a.confidence.upper()}

{source}

---
"""
    display(Markdown(md))


report(answer)


---
## 🚴 Who is currently leading the WorldTour standings?

Tadej Pogačar (UAE Team Emirates - XRG) is currently leading the UCI Individual WorldTour standings by a commanding margin. The Slovenian superstar has accumulated 11,630 points, putting him well clear of second-placed Jonas Vingegaard (Team Visma | Lease a Bike) on 6,885 points. Isaac Del Toro, also of UAE Team Emirates - XRG, sits third with 6,711 points, underlining the strength of the UAE outfit at the top of the rankings. Pogačar's dominance at the summit is no surprise given his recent exceptional form across multiple race categories.

### 📊 Key Facts
- 1st: Tadej Pogačar (UAE Team Emirates - XRG) — 11,630 points
- 2nd: Jonas Vingegaard (Team Visma | Lease a Bike) — 6,885 points
- 3rd: Isaac Del Toro (UAE Team Emirates - XRG) — 6,711 points
- 7th: Paul Seixas (Decathlon CMA CGM Team) — 3,520 points
- 8th: Wout van Aert (Team Visma | Lease a Bike) — 3,495 points
- Rankings data as of approximately May 7, 2026

### 💡 You might also ask
1. Which team is currently leading the UCI WorldTour team standings?
2. How many races has Pogačar won in 2026 to build such a large points lead?
3. Who are the top riders in the UCI WorldTour standings outside of GC specialists?

**Confidence:** 🟢 HIGH

_procyclingstats.com UCI Individual Rider Rankings (data as of ~07.05.2026)_

---


---
## Step 7 — Full Pipeline

Everything in one function — identical to what `main.py` and the Telegram bot run.

In [14]:
def ask(question: str):
    print(f"[1/3] Planning for: '{question}'...")
    plan = create_plan(question)
    print(f"      {len(plan.steps)} steps — {', '.join(s.tool.value for s in plan.steps)}")

    print("[2/3] Executing steps...")
    findings = execute_plan(plan)
    print(f"      {len(findings)} topics collected")

    print("[3/3] Synthesizing answer...")
    answer = synthesize(question, findings)
    print(f"      Confidence: {answer.confidence}\n")

    report(answer)
    return answer


# Try it
result = ask("What are Tadej Pogacar's biggest wins in 2026?")

[1/3] Planning for: 'What are Tadej Pogacar's biggest wins in 2026?'...
      1 steps — pcs_rider_results
[2/3] Executing steps...
      1 topics collected
[3/3] Synthesizing answer...
      Confidence: medium




---
## 🚴 What are Tadej Pogacar's biggest wins in 2026?

Tadej Pogačar has been absolutely dominant in 2026, adding several Monument and Monument-level victories to his already legendary palmares. He opened his season with a stunning 78km solo attack to win Strade Bianche, then claimed Milano-Sanremo (his first career win at La Classicissima), followed by Ronde van Vlaanderen for a second consecutive year. He then added Liège-Bastogne-Liège to his 2026 tally, and continued his dominance into stage racing by winning the Tour de Romandie, where he took at least two individual stage victories (Stage 1 and Stage 2, the latter memorably won in a bunch sprint). This run of victories further cements his status as arguably the most complete and dominant rider in the modern era.

### 📊 Key Facts
- Strade Bianche 2026 – Won with a 78km solo attack on season debut
- Milano-Sanremo 2026 – Monument victory (1.UWT)
- Ronde van Vlaanderen 2026 – Won for the second consecutive year (2025 & 2026)
- Liège-Bastogne-Liège 2026 – Monument victory (also won in 2021, 2024, 2025)
- Tour de Romandie 2026 – Stage victories on Stage 1 (mountain finish at Leysin) and Stage 2 (bunch sprint win)
- Pogacar now has multiple Monuments in the same year, continuing a trend from 2024-25

### 💡 You might also ask
1. Is Pogacar racing the 2026 Tour de France, and who are his main rivals?
2. How does Pogacar's 2026 Monument record compare to his previous seasons?
3. Who finished second in Liège-Bastogne-Liège 2026?

**Confidence:** 🟡 MEDIUM

_procyclingstats.com, cyclingnews.com, cyclinguptodate.com, and web search results_

---


---
## Experiment Zone

Ideas to explore:

1. **Ask about a live race** — Claude will use `firecrawl` for "who is leading right now" type questions
2. **Ask about an obscure rider** — confidence drops to `medium` or `low`
3. **Ask a stage question** — watch the planner pick `pcs_stage` with the right slug/year/stage format
4. **Inspect raw PCS data** before and after synthesis — see how much Claude extracts vs ignores
5. **Edit the synthesizer system prompt** in `agent/synthesizer.py` — change the tone (e.g. more technical, shorter answers)
6. **Trigger Firecrawl directly** — pass a procyclingstats.com URL to `scrape_live_page()` and compare the raw markdown vs what a Tavily search returns for the same race

In [17]:
# Free sandbox — ask anything
ask("Who won stage 5 of the Giro d'Italia 2026?")

[1/3] Planning for: 'Who won stage 5 of the Giro d'Italia 2026?'...
      1 steps — pcs_stage
[2/3] Executing steps...
      1 topics collected
[3/3] Synthesizing answer...
      Confidence: high




---
## 🚴 Who won stage 5 of the Giro d'Italia 2026?

Spain's **Igor Arrieta** (UAE Team Emirates-XRG) won a dramatic and chaotic Stage 5 of the 2026 Giro d'Italia, a 203km rain-soaked route from Praia a Mare to Potenza. Despite crashing with 14km to go and losing over 30 seconds to breakaway companion Afonso Eulálio, Arrieta remounted on a spare bike and hunted the Portuguese rider down. In a remarkable finale, Arrieta nearly crashed again inside the final kilometre before overhauling an exhausted Eulálio in the last 100 metres to take the stage win. Eulálio, however, claimed the consolation prize of the **maglia rosa** (pink jersey), taking over the overall lead in the GC standings after his courageous effort.

### 📊 Key Facts
- Stage 5: Praia a Mare › Potenza (203km)
- Stage winner: Igor Arrieta (UAE Team Emirates-XRG)
- Arrieta crashed with ~14km to go, lost 30+ seconds, remounted on a spare bike
- Arrieta also went off-route in the finale but still managed to win
- Afonso Eulálio (Portugal) finished 2nd but took the pink jersey / GC lead
- Stage was raced in wet, rainy conditions
- Both riders were part of a breakaway that survived to contest the finish

### 💡 You might also ask
1. Who won stage 6 of the Giro d'Italia 2026?
2. What is the overall GC standings after stage 5 of the 2026 Giro d'Italia?
3. How did Igor Arrieta perform overall in the 2026 Giro d'Italia?

**Confidence:** 🟢 HIGH

_Multiple sources including BBC Sport, Rouleur, cyclinguptodate.com, and procyclingstats.com_

---


CyclingAnswer(question="Who won stage 5 of the Giro d'Italia 2026?", answer="Spain's **Igor Arrieta** (UAE Team Emirates-XRG) won a dramatic and chaotic Stage 5 of the 2026 Giro d'Italia, a 203km rain-soaked route from Praia a Mare to Potenza. Despite crashing with 14km to go and losing over 30 seconds to breakaway companion Afonso Eulálio, Arrieta remounted on a spare bike and hunted the Portuguese rider down. In a remarkable finale, Arrieta nearly crashed again inside the final kilometre before overhauling an exhausted Eulálio in the last 100 metres to take the stage win. Eulálio, however, claimed the consolation prize of the **maglia rosa** (pink jersey), taking over the overall lead in the GC standings after his courageous effort.", data_points=['Stage 5: Praia a Mare › Potenza (203km)', 'Stage winner: Igor Arrieta (UAE Team Emirates-XRG)', 'Arrieta crashed with ~14km to go, lost 30+ seconds, remounted on a spare bike', 'Arrieta also went off-route in the finale but still managed t

In [ ]:
ask("Show me the top 10 UCI WorldTour team rankings")

In [ ]:
ask("What is Remco Evenepoel's riding style and what races suit him best?")

In [15]:
# Firecrawl sandbox — this triggers the firecrawl tool in the full pipeline
# Try: "who is currently leading the Giro?", "what happened in today's stage?", "show me the live standings"
ask("Who is currently leading the Giro d'Italia 2026 Stage 10?")

[1/3] Planning for: 'Who is currently leading the Giro d'Italia 2026 Stage 10?'...
      2 steps — firecrawl, firecrawl
[2/3] Executing steps...
      2 topics collected
[3/3] Synthesizing answer...
      Confidence: low




---
## 🚴 Who is currently leading the Giro d'Italia 2026 Stage 10?

Stage 10 of the Giro d'Italia 2026 is a 42 km Individual Time Trial (ITT) running from Viareggio to Massa, scheduled for May 19, 2026 with a start time of 13:15. At the time of this query, no results are yet available for Stage 10 — the stage winner and live leaderboard data are currently empty, suggesting the stage either has not yet started or results have not been published to the data source. Heading into this crucial ITT, Jonas Vingegaard (winner of Stages 7 and 9) and Jhonatan Narváez (winner of Stages 4 and 8) have been the standout performers of the race so far. As a time trial specialist environment, Vingegaard — known for his exceptional TT abilities — will be among the favourites to take the stage win and potentially consolidate or extend his GC position.

### 📊 Key Facts
- Stage 10 is a 42 km ITT from Viareggio to Massa on May 19, 2026
- No stage 10 results are currently available from the data source
- Start time: 13:15
- Jonas Vingegaard won Stages 7 and 9 heading into Stage 10
- Jhonatan Narváez won Stages 4 and 8
- Other stage winners: Paul Magnier (Stages 1 & 3), Guillermo Thomas Silva (Stage 2), Igor Arrieta (Stage 5), Davide Ballerini (Stage 6)
- Stage profile score: 2, with only 105 vertical meters — flat ITT terrain

### 💡 You might also ask
1. Who won the Giro d'Italia 2026 Stage 10 ITT?
2. What is the overall GC standings after Stage 9 of Giro d'Italia 2026?
3. How has Jonas Vingegaard been performing at the 2026 Giro d'Italia?

**Confidence:** 🔴 LOW

_procyclingstats.com — Stage 10 results table was empty at time of data fetch_

---


CyclingAnswer(question="Who is currently leading the Giro d'Italia 2026 Stage 10?", answer="Stage 10 of the Giro d'Italia 2026 is a 42 km Individual Time Trial (ITT) running from Viareggio to Massa, scheduled for May 19, 2026 with a start time of 13:15. At the time of this query, no results are yet available for Stage 10 — the stage winner and live leaderboard data are currently empty, suggesting the stage either has not yet started or results have not been published to the data source. Heading into this crucial ITT, Jonas Vingegaard (winner of Stages 7 and 9) and Jhonatan Narváez (winner of Stages 4 and 8) have been the standout performers of the race so far. As a time trial specialist environment, Vingegaard — known for his exceptional TT abilities — will be among the favourites to take the stage win and potentially consolidate or extend his GC position.", data_points=['Stage 10 is a 42 km ITT from Viareggio to Massa on May 19, 2026', 'No stage 10 results are currently available from